In [6]:
import os
import sys

repo_root_path = os.path.abspath(os.path.join("..", "..", ".."))

if repo_root_path not in sys.path:
    sys.path.append(repo_root_path)


In [7]:
from datetime import datetime, timedelta

import requests

from dotenv import load_dotenv

from quotaclimat.data_processing.mediatree.config import (
    get_user,
    get_password,
    get_auth_url,
)

load_dotenv()

PASSWORD = get_password()
AUTH_URL = get_auth_url()
USER = get_user()
MEDIATREE_API_URL = os.environ.get("MEDIATREE_API_URL")


def get_auth_token():
    post_arguments = {
        "grant_type": "password",
        "username": USER,
        "password": PASSWORD,
    }
    assert AUTH_URL is not None, "AUTH_URL is not set"
    response = requests.post(AUTH_URL, data=post_arguments)
    output = response.json()
    token = output["data"]["access_token"]
    return token


def get_single_export(
    token: str, channel: str, from_date: datetime, to_date: datetime, media_format: str
):
    response = requests.get(
        f"{MEDIATREE_API_URL}/export/single",
        params={
            "token": token,
            "channel": channel,
            "cts_in": int(from_date.timestamp()),
            "cts_out": int(to_date.timestamp()),
            "media_format": media_format,
        },
    )
    return response.json()


def _download_export(token, file_name, channel: str, from_date: datetime, to_date: datetime):
    single_export = get_single_export(token, channel, from_date, to_date, "mp3")
    file_src = single_export["src"]

    response = requests.get(file_src)

    os.makedirs(os.path.dirname(file_name), exist_ok=True)
    with open(file_name, "wb") as f:
        f.write(response.content)

EXPORTS_FOLDER = "../exports"

def download_export(token, channel: str, from_date: datetime, to_date: datetime):
    file_name = f"{channel}_{from_date.strftime('%Y-%m-%d_%H-%M-%S')}_{to_date.strftime('%Y-%m-%d_%H-%M-%S')}.mp3"

    file_path = os.path.join(EXPORTS_FOLDER, file_name)

    if not os.path.isfile(file_path):
        print(f"Downloading export for {channel} from {from_date} to {to_date}...")
        _download_export(token, file_path, channel, from_date, to_date)

    return file_path

def all_intervals_between(
    start_date: datetime, end_date: datetime, interval: timedelta
):
    intervals = []
    current_start = start_date
    while current_start < end_date:
        current_end = min(current_start + interval, end_date)
        intervals.append((current_start, current_end))
        current_start = current_end
    return intervals

def export_channel_whole_week(token, channel: str, week_start_date: datetime):
    for start_date, end_date in all_intervals_between(
        week_start_date, week_start_date + timedelta(days=7), timedelta(hours=1)
    ):
        download_export(token, channel, start_date, end_date)


In [8]:
AUTH_TOKEN = get_auth_token()

get_single_export(
    AUTH_TOKEN,
    channel="tf1",
    from_date=datetime(2025, 3, 4, 12, 0, 0),
    to_date=datetime(2025, 3, 4, 13, 0, 0),
    media_format="mp4",
)


{'src': 'https://vod.mediatree.fr/assets/tf1_2025-03-04T12-00-00Z_2025-03-04T13-00-00Z.mp4',
 'cts_in': 1741089600,
 'cts_out': 1741093200,
 'elapsed_time_ms': 124}

In [9]:
AUTH_TOKEN = get_auth_token()

_download_export(
    AUTH_TOKEN,
    "/root/Workspace/quotaclimat/analyse/advertising/exports/00_small.mp3",
    "tf1",
    datetime(2025, 3, 4, 12, 0, 0),
    datetime(2025, 3, 4, 12, 5, 0),
)

In [10]:
AUTH_TOKEN = get_auth_token()

MONDAY_LAST_YEAR = datetime(2025, 3, 3)

export_channel_whole_week(AUTH_TOKEN, "tf1", MONDAY_LAST_YEAR)
export_channel_whole_week(AUTH_TOKEN, "france2", MONDAY_LAST_YEAR)
export_channel_whole_week(AUTH_TOKEN, "fr3-idf", MONDAY_LAST_YEAR)
export_channel_whole_week(AUTH_TOKEN, "m6", MONDAY_LAST_YEAR)
